# BCUL importer Debug for new batches

### Imports

In [1]:
# Automatically reloads modules when you make changes (useful during development)
%load_ext autoreload 
%autoreload 2

In [2]:
import logging
import os
import json
import string
from collections import namedtuple
import sys
import tqdm

from dask import bag as db



In [3]:
from impresso_essentials.utils import ALL_MEDIA

In [4]:
print(ALL_MEDIA)

['AATA', 'ABal', 'ACI', 'AChal', 'AGE52', 'AGMO', 'AHEC', 'ALBN', 'ALST', 'ALT', 'ANJO', 'ANWT', 'ARB', 'ARGB', 'AS', 'AUBO', 'AV', 'BBLT', 'BCE1', 'BCL2', 'BDC', 'BDPO', 'BEHI', 'BELL', 'BFNP', 'BGCH', 'BGFP', 'BGJO', 'BHCH', 'BHFA', 'BIST', 'BKNW', 'BLB', 'BLHD', 'BLMY', 'BLOT', 'BLSD', 'BLWJ', 'BNER', 'BNN', 'BNPT', 'BNWL', 'BPDH', 'BPHF', 'BQGA', 'BRAD', 'BRBN', 'BREM', 'BREN', 'BRGA', 'BRIF', 'BRLB', 'BRLU', 'BRMG', 'BRMW', 'BRNP', 'BROR', 'BRPR', 'BRPT', 'BRSS', 'BRST', 'BRTB', 'BTEP', 'BWNW', 'BWTE', 'Bombe', 'CBEP', 'CCEX', 'CCGZ', 'CCWA', 'CDV', 'CFCE', 'CFTM', 'CGFG', 'CGGA', 'CHOR', 'CHPL', 'CHPN', 'CHSO', 'CHTI', 'CHTR', 'CHTT', 'CHU', 'CICN', 'CKTC', 'CL', 'CLDF', 'CLN', 'CLNW', 'CLTP', 'CLib', 'CMCH', 'CMGA', 'CMSN', 'CNMR', 'CNSN', 'COGE', 'CON', 'COUR', 'CPAD', 'CREC', 'CRWN', 'CSMP', 'CSTT', 'CWPG', 'CWPR', 'Cancoire', 'Castigat', 'Charivari', 'CharivariCH', 'Croquis', 'DCEA', 'DCWR', 'DDEN', 'DDIS', 'DETO', 'DFS', 'DGMH', 'DHEX', 'DJWN', 'DLE', 'DNLN', 'DP', 'DPLT', '

In [5]:
from text_preparation.importers.detect import _apply_datefilter
from text_preparation.importers.bcul.helpers import parse_date, find_mit_file
from text_preparation.importers.bcul.classes import BculNewspaperIssue

In [6]:
from text_preparation.importers.detect import _apply_datefilter
from text_preparation.importers.bcul.helpers import parse_date, find_mit_file
from text_preparation.importers.bcul.classes import BculNewspaperIssue


logger = logging.getLogger(__name__)

BculIssueDir = namedtuple(
    "IssueDirectory", ["provider", "alias", "date", "edition", "path", "mit_file_type"]
)

In [7]:
BASE_DIR = "/mnt/project_impresso/original/BCUL"

In [8]:
# OLD_ALIASES_FILEPATH = '/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases1_2.json'
ALIASES_FILEPATH = '/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases.json'


## Detect


In [ ]:
def dir2issue(path: str, journal_info: dict[str, str]) -> BculIssueDir | None:
    """Create a `BculIssueDir` object from a directory.

    Note:
        This function is called internally by `detect_issues`

    Args:
        path (str): The path of the issue.
        access_rights (dict): Dictionary for access rights.

    Returns:
        BculIssueDir | None: New `BculIssueDir` object.
    """
    mit_file = find_mit_file(path)
    if mit_file is None:
        logger.error("Could not find MIT file in %s", path)
        return None

    mit_ext = mit_file.split(".")[-1]
    expected_ext = journal_info["mit_file_type"]
    print('mit file ends with:', mit_file, mit_ext, expected_ext)
    # --- handle 'both' case --- 
    if expected_ext == "both":
        if mit_ext not in ('xml', 'json'):
            logger.warning(
                "Found mit file %s has unexpected extension %s, expected 'xml' or 'json'",
                os.path.join(path, mit_file),
                mit_ext,
            )
            # accept either format without changing journal_info
    else: 
        # --- normal case ---
        if not mit_file.endswith(journal_info["mit_file_type"]):
            logger.warning(
                "Found mit file %s does not correspond to mit file type %s",
                os.path.join(path, mit_file),
                expected_ext,
            )
            # override the mit file type if the extension of the file found does not match
            journal_info["mit_file_type"] = mit_ext

    date = parse_date(mit_file)

    # check if multiple issues are at this date:
    day_dir = os.path.dirname(path)
    day_editions = list(os.listdir(day_dir))
    day_editions = [
        str(i)
        for i in os.listdir(day_dir)
        if i != ".DS_Store"
    ]

    if len(day_editions) > 1:
        # if multiple issues exist for a given day, find the correct edition
        logger.info("Multiple issues for %s, finding the edition", day_dir)
        # exclude incorrect issues from the list
        index = sorted(day_editions).index(os.path.basename(path))
        edition = string.ascii_lowercase[index]
    else:
        edition = "a"

    return BculIssueDir(
        provider="BCUL",
        alias=journal_info["alias"],
        date=date,
        edition=edition,
        path=path,
        mit_file_type=mit_ext if expected_ext == "both" else journal_info["mit_file_type"],
    )


## DO NOT RUN IT TAKES FOREVER

In [ ]:
# open and read bcul_alias.json file
with open(ALIASES_FILEPATH, "rb") as f:
    alias_mapping = json.load(f)

dir_path, dirs, files = next(os.walk(BASE_DIR))

journal_dirs = [
    os.path.join(dir_path, _dir)
    for _dir in dirs
    if _dir not in ["OLD", "wrong_BCUL", ".DS_Store"] and _dir in alias_mapping
]
issue_dirs = []
for journal in journal_dirs:
    logger.info("Detecting issues for %s.", journal)
    for dir_path, dirs, files in os.walk(journal):
        title = journal.split("/")[-1]
        # check if we are in the directory of a (valid) issue
        if (
            len(files) > 1
            and "solr" not in dir_path
        ):
            issue_dirs.append(dir2issue(dir_path, alias_mapping[title]))

# return issue_dirs

In [8]:
journal = "/mnt/project_impresso/original/BCUL/Domaine_Public"

In [88]:
issue_dirs = []
nb_issues = 0
for dir_path, dirs, files in os.walk(journal):
    title = journal.split("/")[-1]
    if (
            len(files) > 1
            and "solr" not in dir_path
        ):
            nb_issues += 1
            issue_dirs.append(dir2issue(dir_path, alias_mapping[title]))


    

## bcul.classes.py

In [8]:
# open and read bcul_alias.json file
with open(OLD_ALIASES_FILEPATH, "rb") as f:
    old_alias_mapping = json.load(f)

In [11]:
# open and read bcul_alias.json file
with open(ALIASES_FILEPATH, "rb") as f:
    alias_mapping = json.load(f)

In [20]:
MEdir = dir2issue("/mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522", alias_mapping["Le_Grelot"])
CONFdir = dir2issue("/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437", alias_mapping["Confiance"])
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])

mit file ends with: /mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522/Grelot_0012_1845_09_01_0001_mit.xml xml xml
mit file ends with: /mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437/EM_1950_09_00_mit.json json json
mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both


In [25]:
from text_preparation.importers.bcul.detect import dir2issue


In [ ]:
ACIdir = dir2issue("/mnt/project_impresso/original/BCUL/Almanach_pour_le_commerce/1832/01/01/171722", alias_mapping["Almanach_pour_le_commerce"])

In [ ]:
CONFdir = dir2issue("/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437", alias_mapping["Confiance"])


In [14]:
issue = BculNewspaperIssue(DPdir)
for p in issue.pages:
    print(p.page_data["id"], p.page_data.get("fw"), p.page_data.get("fh"))


il y a :8 pages
DP-1987-10-15-a-p0001 2256 3015
DP-1987-10-15-a-p0002 2256 3015
DP-1987-10-15-a-p0003 2256 3015
DP-1987-10-15-a-p0004 2256 3015
DP-1987-10-15-a-p0005 2256 3015
DP-1987-10-15-a-p0006 2256 3015
DP-1987-10-15-a-p0007 2256 3015
DP-1987-10-15-a-p0008 2256 3015


In [42]:
import requests, json
url = "https://scriptorium.bcu-lausanne.ch/api/iiif/168346/manifest"
response = requests.get(url, verify=False)
print(response.status_code)

200


In [43]:
file_path = '/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437/6355589_exif.json'
with open(file_path, 'r', encoding='utf-8') as jf:
    exif_list = json.load(jf)


In [24]:
exif_data = exif_list[0]
jpeg_info = exif_data.get("Jpeg2000", {})

In [25]:
w = jpeg_info.get('ImageWidth')
h = jpeg_info.get('ImageHeight')

### Inspect content items of one issue

In [44]:
MEdir = dir2issue("/mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522", alias_mapping["Le_Grelot"])
CONFdir = dir2issue("/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437", alias_mapping["Confiance"])
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])

mit file ends with: /mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522/Grelot_0012_1845_09_01_0001_mit.xml xml xml
mit file ends with: /mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437/EM_1950_09_00_mit.json json json
mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both


In [69]:
from text_preparation.importers.bcul.classes import BculNewspaperIssue


In [70]:
issue = BculNewspaperIssue(DPdir)

il y a :8 pages


In [17]:
for ci in issue.content_items:
    print("CI ID:", ci["m"]["id"])
    print("Type:", ci["m"]["tp"])
    print("Pages:", ci["m"]["pp"])
    print('Reading Order', ci['m']['ro'])
    print("Legacy info:", ci.get("l"))
    print("IIIF link:", ci["m"].get("iiif_link"))
    print("-" * 50)

CI ID: DP-1987-10-15-a-i0001
Type: page
Pages: [1]
Reading Order 1
Legacy info: {'issue_id': 169220, 'page_id': 1782642, 'parts': [{'comp_role': 'text blocks', 'comp_id': ['{B88CB401-BAB8-4530-8659-27F79A700C8B}', '{BD734612-6B7F-4055-9116-3FEA0C8F077C}', '{4BC1705A-0314-4F8D-90F8-141CA0F808D8}', '{AD293151-9AA8-41AB-95D6-E86EA9D5306A}', '{28C8A244-6AB5-4B0B-9366-C2DB7B282EE6}'], 'comp_fileid': 'DP_0879_1987_10_15_01_page_1.xml', 'comp_page_no': 1}], 'source': 'DP_0879_1987_10_15_01_mit.xml'}
IIIF link: None
--------------------------------------------------
CI ID: DP-1987-10-15-a-i0009
Type: image
Pages: [1]
Reading Order 2
Legacy info: {'issue_id': 169220, 'page_id': 1782642, 'parts': [{'comp_role': 'image', 'comp_id': '{91A2C083-D8FC-4AFC-9059-2038E51A72BC}', 'comp_fileid': 'DP_0879_1987_10_15_01_page_1.xml', 'comp_page_no': 1, 'coords': [57, 61, 588, 1316]}], 'source': 'DP_0879_1987_10_15_01_mit.xml'}
IIIF link: https://www.scriptorium.ch/api/iiif-img/v3/1782642/info.json
---------

In [17]:
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])

mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both


## Legacy

In [ ]:
# create issue object
from collections import Counter
import json

# --- 1. Load issue ---
issue = BculNewspaperIssue(DPdir)

# --- 2. Overview ---
types = [ci["m"]["tp"] for ci in issue.content_items]
print("Counts by CI type:", Counter(types))
print(f"Total CIs: {len(issue.content_items)}\n")

# --- 3. Inspect first examples by type ---
def show_example(ci_type, max_show=2):
    cis = [ci for ci in issue.content_items if ci["m"]["tp"] == ci_type]
    if not cis:
        print(f"No content items of type '{ci_type}' found.\n")
        return
    print(f"🔍 Example {ci_type} CI ({len(cis)} total):\n")
    for ci in cis[:max_show]:
        print(json.dumps(ci["m"], indent=2))
        print("Legacy:")
        print(json.dumps(ci["l"], indent=2))
        if "c" in ci:
            print("Coordinates:", ci["c"])
        print("-" * 80)
    print()

show_example("page")
show_example("image")
show_example("table")

# --- 4. Sanity checks ---
print("Running consistency checks...")

# Check that all required fields exist
for ci in issue.content_items:
    assert "m" in ci, "Missing 'm' section!"
    assert "l" in ci, f"Missing 'l' (legacy) section in CI {ci['m']['id']}"
    assert "id" in ci["m"], "Missing m.id!"
    assert ci["m"]["pp"], f"Missing page number (pp) in CI {ci['m']['id']}"

# Check that all CIs have a valid page legacy id
for ci in issue.content_items:
    lid = ci["l"].get("id")
    assert lid, f"Missing legacy id in CI {ci['m']['id']}"

# Check that all images/tables share the same legacy id as their page
page_legacy_ids = {
    ci["m"]["pp"][0]: ci["l"]["id"]
    for ci in issue.content_items if ci["m"]["tp"] == "page"
}
for ci in issue.content_items:
    # correct relationship check
    if ci["m"]["tp"] in {"image", "table"}:
        pnum = ci["m"]["pp"][0]
        # make sure it belongs to a valid page and comp_id exists
        assert "comp_id" in ci["l"]["parts"][0], f"Missing comp_id in {ci['m']['id']}"
        assert ci["l"]["parts"][0]["comp_page_no"] == pnum, f"CI {ci['m']['id']} has wrong page reference"

        
# Optional: check for missing comp_id
missing_comp_ids = [
    ci["m"]["id"] for ci in issue.content_items
    if not ci["l"]["parts"][0].get("comp_id")
]

# print which content items are missing comp_id
if missing_comp_ids:
    print(f"  {len(missing_comp_ids)} CIs have no comp_id in legacy parts:")
    for cid in missing_comp_ids:
        ci = next(c for c in issue.content_items if c["m"]["id"] == cid)
        print(f"  - CI ID: {cid} (id: {ci['l']['id']}, type: {ci['m']['tp']}, pages: {ci['m']['pp']})")
else:
    print("All CIs have comp_id values.")

print("All sanity checks passed successfully.\n")


# --- 5. Summary per page ---
print("Pages summary (page_no → CI types):")
from collections import defaultdict
page_summary = defaultdict(list)
for ci in issue.content_items:
    tp = ci["m"]["tp"]
    for p in ci["m"]["pp"]:
        page_summary[p].append(tp)
for p in sorted(page_summary):
    print(f"Page {p}: {Counter(page_summary[p])}")


🗞️  Counts by CI type: Counter({'page': 8, 'image': 7, 'table': 4})
Total CIs: 19

🔍 Example page CI (8 total):

{
  "id": "DP-1987-10-15-a-i0001",
  "pp": [
    1
  ],
  "tp": "page",
  "ro": 1
}
Legacy:
{
  "id": "DP-1987-10-15-a-p0001",
  "parts": [
    {
      "comp_role": "page",
      "comp_id": "DP-1987-10-15-a-p0001",
      "comp_fileid": "DP_0879_1987_10_15_01_page_1.xml",
      "comp_page_no": 1
    }
  ],
  "source": {
    "mit": "DP_0879_1987_10_15_01_mit.xml",
    "page_xml": [
      "DP_0879_1987_10_15_01_page_1.xml"
    ],
    "page_image": [
      "https://www.scriptorium.ch/api/iiif-img/v3/1782642"
    ]
  }
}
--------------------------------------------------------------------------------
{
  "id": "DP-1987-10-15-a-i0002",
  "pp": [
    2
  ],
  "tp": "page",
  "ro": 3
}
Legacy:
{
  "id": "DP-1987-10-15-a-p0002",
  "parts": [
    {
      "comp_role": "page",
      "comp_id": "DP-1987-10-15-a-p0002",
      "comp_fileid": "DP_0879_1987_10_15_01_page_2.xml",
      "comp_

In [ ]:
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])
issue = BculNewspaperIssue(DPdir)

# Test snippet: check if issue_id and page_id were correctly added

# Pick the first 3 content items for quick inspection
for ci in issue.content_items[:3]:
    legacy = ci.get("l", {})
    m = ci.get("m", {})
    print(f"CI ID: {m.get('id')}  (type: {m.get('tp')})")
    print(f"   → Issue ID: {legacy.get('issue_id')}")
    print(f"   → Page ID: {legacy.get('page_id')}")
    print(f"   → Page number: {m.get('pp')}")
    print(f"   → Source XML: {legacy.get('source', {}).get('page_xml')}")
    print("-" * 80)

# Optional: sanity checks
issue_ids = {ci["l"].get("issue_id") for ci in issue.content_items}
page_ids = [ci["l"].get("page_id") for ci in issue.content_items if ci["l"].get("page_id")]

print(f"Unique issue IDs found: {issue_ids}")
print(f"Example of page IDs: {page_ids[:5]}")


mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both
📰 CI ID: DP-1987-10-15-a-i0001  (type: page)
   → Issue ID: 169220
   → Page ID: 1782642
   → Page number: [1]
   → Source XML: ['DP_0879_1987_10_15_01_page_1.xml']
--------------------------------------------------------------------------------
📰 CI ID: DP-1987-10-15-a-i0002  (type: image)
   → Issue ID: 169220
   → Page ID: 1782642
   → Page number: [1]
   → Source XML: ['DP_0879_1987_10_15_01_page_1.xml']
--------------------------------------------------------------------------------
📰 CI ID: DP-1987-10-15-a-i0003  (type: page)
   → Issue ID: 169220
   → Page ID: 1782643
   → Page number: [2]
   → Source XML: ['DP_0879_1987_10_15_01_page_2.xml']
--------------------------------------------------------------------------------
✅ Unique issue IDs found: {169220}
✅ Example of page IDs: [1782642, 1782642, 1782643, 1782643, 1782644]


In [ ]:
print(f"Issue ID: {issue.id}")
print(f"Total content items: {len(issue.content_items)}\n")

for ci in issue.content_items:
    ci_id = ci.get("m", {}).get("id", "UNKNOWN_CI")
    l_id = ci.get("l", {}).get("file_id", "UNKNOWN_LEGACY_ID")
    ci_type = ci.get("m", {}).get("tp", "UNKNOWN_TYPE")
    issue_id = ci.get("l", {}).get("issue_id", "MISSING")
    page_id = ci.get("l", {}).get("page_id", "MISSING")
    parts = ci.get("l", {}).get("parts", [])
    comp_ids = [p.get("comp_id", "MISSING") for p in parts]
    print(f"- CI ID: {ci_id} | File_id: {l_id} | Type: {ci_type} | issue_id: {issue_id} | page_id: {page_id} | comp_ids: {comp_ids}")

print("\nDone listing all comp_ids for this issue.")


📰 Issue ID: DP-1987-10-15-a
Total content items: 19

- CI ID: DP-1987-10-15-a-i0001 | File_id: DP_0879_1987_10_15_01_page_1 | Type: page | issue_id: 169220 | page_id: 1782642 | comp_ids: [['{B88CB401-BAB8-4530-8659-27F79A700C8B}', '{BD734612-6B7F-4055-9116-3FEA0C8F077C}', '{4BC1705A-0314-4F8D-90F8-141CA0F808D8}', '{AD293151-9AA8-41AB-95D6-E86EA9D5306A}', '{28C8A244-6AB5-4B0B-9366-C2DB7B282EE6}']]
- CI ID: DP-1987-10-15-a-i0002 | File_id: DP_0879_1987_10_15_01_page_1 | Type: image | issue_id: 169220 | page_id: 1782642 | comp_ids: ['{91A2C083-D8FC-4AFC-9059-2038E51A72BC}']
- CI ID: DP-1987-10-15-a-i0003 | File_id: DP_0879_1987_10_15_01_page_2 | Type: page | issue_id: 169220 | page_id: 1782643 | comp_ids: [['{53951ED6-9AA6-4DB5-83D3-450FEE3A2CAD}', '{83FDCCAF-11B5-4A70-A299-D54E8C2AF58A}']]
- CI ID: DP-1987-10-15-a-i0004 | File_id: DP_0879_1987_10_15_01_page_2 | Type: table | issue_id: 169220 | page_id: 1782643 | comp_ids: ['{F93C713D-66EB-432A-BF96-37D2D3288F2C}']
- CI ID: DP-1987-10-15-

In [28]:
filename = "123456.xml"
page_id = os.path.splitext(filename)[0]

In [29]:
page_id

'123456'

### Hyphenisation

In [20]:
from bs4 import BeautifulSoup
from text_preparation.importers.bcul.helpers import parse_char_tokens

with open("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_page_1.xml", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "xml")

for line in soup.find_all("line"):
    tokens = parse_char_tokens(line.find_all("charParams"))
    print(tokens)


[{'c': [618, 2018, 23, 18], 'tx': 'o'}]
[{'c': [618, 2038, 22, 32], 'tx': 'E'}]
[{'c': [565, 2110, 32, 16], 'tx': 'OC'}, {'c': [615, 2102, 25, 24], 'tx': 'Ì3'}]
[{'c': [574, 2140, 76, 48], 'tx': 'Si'}]
[{'c': [575, 2197, 22, 20], 'tx': 'o'}, {'c': [614, 2171, 36, 53], 'tx': 'J'}]
[{'c': [574, 2232, 23, 17], 'tx': 'U'}, {'c': [618, 2228, 30, 19], 'tx': 'W)'}]
[{'c': [574, 2250, 23, 21], 'tx': 'O'}, {'c': [618, 2250, 21, 19], 'tx': 'C'}]
[{'c': [565, 2285, 75, 28], 'tx': '!2>'}]
[{'c': [564, 2397, 29, 23], 'tx': 'a'}]
[{'c': [564, 2424, 28, 29], 'tx': 'C'}, {'c': [607, 2434, 34, 20], 'tx': '73'}]
[{'c': [565, 2457, 27, 23], 'tx': 'c'}, {'c': [618, 2456, 23, 22], 'tx': 'c'}]
[{'c': [564, 2489, 30, 25], 'tx': 'c5'}, {'c': [618, 2480, 23, 43], 'tx': 'g'}]
[{'c': [565, 2516, 29, 19], 'tx': 'in'}, {'c': [618, 2508, 23, 30], 'tx': 'b'}]
[{'c': [566, 2540, 28, 27], 'tx': '3'}, {'c': [618, 2541, 24, 28], 'tx': 'P'}]
[{'c': [565, 2572, 29, 24], 'tx': 'C3'}]
[{'c': [553, 2617, 89, 26], 'tx': '►J.S

In [23]:
from bs4 import BeautifulSoup
from text_preparation.importers.bcul.helpers import parse_textblock

xml_path = "/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_page_1.xml"
xml_path ='/mnt/project_impresso/original/BCUL/Chut/1977/01/01/126474/Chut_0010_1977_00_00_0001_page_3.xml'
with open(xml_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "xml")

# pick a block that we know contains the hyphen case (inspect manually if needed)
for block in soup.find_all("block", {"blockType": "Text"}):
    region = parse_textblock(block, "DUMMY_PAGE_CI")
    # print tokens lines and look for 'hy'/'nf'
    for para in region["p"]:
        for line in para["l"]:
            toks = line["t"]
            print([ { "tx": t.get("tx"), "hy": t.get("hy"), "nf": t.get("nf") } for t in toks ])
    print("---- block end ----")


[{'tx': 'CH', 'hy': None, 'nf': None}, {'tx': 'ut!', 'hy': None, 'nf': None}, {'tx': 'RESPECTE', 'hy': None, 'nf': None}]
[{'tx': 'LA', 'hy': None, 'nf': None}, {'tx': 'TRÊVE', 'hy': None, 'nf': None}, {'tx': 'DE', 'hy': None, 'nf': None}, {'tx': 'NOËL', 'hy': None, 'nf': None}]
[{'tx': 'Disons-le', 'hy': None, 'nf': None}, {'tx': 'tout', 'hy': None, 'nf': None}, {'tx': 'net:', 'hy': None, 'nf': None}, {'tx': 'on', 'hy': None, 'nf': None}, {'tx': 'a', 'hy': None, 'nf': None}, {'tx': 'envie', 'hy': None, 'nf': None}, {'tx': 'de', 'hy': None, 'nf': None}, {'tx': 'prendre', 'hy': None, 'nf': None}, {'tx': 'des', 'hy': None, 'nf': None}, {'tx': 'vacances!', 'hy': None, 'nf': None}, {'tx': 'Ce', 'hy': None, 'nf': None}, {'tx': 'n’est', 'hy': None, 'nf': None}]
[{'tx': 'pas', 'hy': None, 'nf': None}, {'tx': 'que', 'hy': None, 'nf': None}, {'tx': 'l’amitié', 'hy': None, 'nf': None}, {'tx': 'que', 'hy': None, 'nf': None}, {'tx': 'nous', 'hy': None, 'nf': None}, {'tx': 'portons', 'hy': None, 'n

[{'tx': '«LÀ', 'hy': None, 'nf': None}, {'tx': 'VILLE', 'hy': None, 'nf': None}, {'tx': 'QUI', 'hy': None, 'nf': None}, {'tx': 'N’EXISTAIT', 'hy': None, 'nf': None}, {'tx': 'PAS»', 'hy': None, 'nf': None}]

[{'tx': 'de', 'hy': None, 'nf': None}, {'tx': 'l’ami-auteur', 'hy': None, 'nf': None}, {'tx': 'Christin', 'hy': None, 'nf': None}, {'tx': 'et', 'hy': None, 'nf': None}, {'tx': 'de', 'hy': None, 'nf': None}, {'tx': 'son', 'hy': None, 'nf': None}, {'tx': 'complice', 'hy': None, 'nf': None}]

[{'tx': 'Bilal', 'hy': None, 'nf': None}, {'tx': 'que', 'hy': None, 'nf': None}, {'tx': 'je-ne-connais-pas-personnelle-', 'hy': True, 'nf': None}]

[{'tx': 'ment-et-qui-dessine-comme-un-chef.', 'hy': None, 'nf': 'je-ne-connais-pas-personnellement-et-qui-dessine-comme-un-chef.'}, {'tx': 'Re¬', 'hy': True, 'nf': None}]

[{'tx': 'marquez', 'hy': None, 'nf': 'Remarquez'}